In [1]:
import re
import numpy as np
import pandas as pd
from pathlib import Path
import lightgbm as lgb
from sklearn.linear_model import Ridge
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

# =========================
# CONFIG
# =========================
ROOT     = Path("/home/mohamed/SDD/hackathon/sia-predicting-short-form-video-popularity")
DATA_DIR = ROOT / "Data"

TRAIN_MERGED = DATA_DIR / "X_train_merged.csv"
TEST_MERGED  = DATA_DIR / "X_test_merged.csv"
Y_PATH       = ROOT / "y_train.csv"

N_SPLITS     = 5
SEED         = 42
PCA_CLIP_DIM  = 64
PCA_VGG_DIM   = 64

# =========================
# COLUMN GROUPS (noms exacts confirmés)
# =========================
CLIP_COLS    = [f"c{i}"          for i in range(512)]
VGG_COLS     = [f"vggish_{i:03d}" for i in range(256)]

LIBROSA_COLS = [
    "rms_mean","rms_std","zcr_mean","zcr_std",
    "centroid_mean","centroid_std","bandwidth_mean","bandwidth_std",
    "rolloff_mean","rolloff_std","tempo","onset_rate",
    # MFCCs 0-5 seulement
    *[f"mfcc_{i}_{s}" for i in range(6) for s in ("mean","std")],
]

META_COLS = [
    "uploader_log_count","uploader_short_log_count","channel_log_count",
    "artist_log_count","track_log_count","album_log_count","uid_log_count",
    "has_music","is_original_sound","resolution_area","is_vertical",
    "is_full_hd","aspect_w_over_h","nb_words","avg_word_length",
    "has_exclamation","has_question","uppercase_ratio","has_mention",
    "has_url","has_number","words_per_second","uploader_unique_artist",
    "uploader_unique_track","deezer_rank","status",
    # texte enrichi
    "text_len","nb_hashtags","nb_emojis","hashtag_density","emoji_density",
    "sentiment_score","sentiment_abs","nb_textbloc",
    "aspect_ratio","video_duration","release_year",
]

VIDEO_COLS = [
    # luminosité (5 frames × 3 stats)
    *[f"f{i}_{s}" for i in range(1,6) for s in ("sharpness","brightness","saturation")],
    # dynamisme
    "hook_motion","hook_shake","content_motion","content_shake","max_peak_motion",
    "avg_motion","camera_shake","duration","num_cuts",
    # attention
    "att_mean_saliency","att_max_saliency","att_min_saliency","att_skewness",
    "att_kurtosis","att_hook_score","att_outro_score","att_peak_location",
    "att_high_attention_ratio","att_attention_instability",
    # vision
    "yolo_person","yolo_ski_snow","avg_face_count","max_face_coverage",
    "body_presence_score",
    # emotions
    "dominant_emotion_angry","dominant_emotion_disgust","dominant_emotion_fear",
    "dominant_emotion_happy","dominant_emotion_neutral","dominant_emotion_none",
    "dominant_emotion_sad","dominant_emotion_surprise",
    # couleurs RGB
    *[f"R{i}" for i in range(1,4)], *[f"G{i}" for i in range(1,4)],
    *[f"B{i}" for i in range(1,4)], *[f"W{i}" for i in range(1,4)],
]

# one-hot uploaders + langue
ONEHOT_COLS_PATTERN = re.compile(r"^(up_|lang_)")

# =========================
# UTILS
# =========================
def read_csv_robust(path):
    try:
        return pd.read_csv(path, sep=None, engine="python")
    except Exception:
        return pd.read_csv(path, sep=",", engine="python")

def normalize_id(s):
    s = s.astype(str).str.strip()
    s = s.str.replace(r"^(VIDEO_|video_|Video_)", "", regex=True)
    s = s.str.extract(r"(\d+)", expand=False).fillna(s)
    return s.str.strip()

def apply_pca(X_tr, X_te, n_comp, name):
    n_comp = min(n_comp, X_tr.shape[1], X_tr.shape[0])
    scaler = StandardScaler()
    Xs_tr  = scaler.fit_transform(np.nan_to_num(X_tr))
    Xs_te  = scaler.transform(np.nan_to_num(X_te))
    pca    = PCA(n_components=n_comp, random_state=SEED)
    tr_out = pca.fit_transform(Xs_tr)
    te_out = pca.transform(Xs_te)
    var    = pca.explained_variance_ratio_.cumsum()[-1]
    print(f"   PCA {name}: {X_tr.shape[1]} → {n_comp} dims | var expliquée: {var:.1%}")
    cols   = [f"{name}_pca_{i}" for i in range(n_comp)]
    return pd.DataFrame(tr_out, columns=cols), pd.DataFrame(te_out, columns=cols)

# =========================
# 1) LOAD + ALIGN
# =========================
print("📂 Chargement...")
train = read_csv_robust(TRAIN_MERGED)
test  = read_csv_robust(TEST_MERGED)
y_df  = read_csv_robust(Y_PATH)

for df in (train, test, y_df):
    if "ID" not in df.columns:
        df.rename(columns={df.columns[0]: "ID"}, inplace=True)
    df["ID"] = normalize_id(df["ID"])

for df in (train, test):
    if "popularity" in df.columns:
        df.drop(columns=["popularity"], inplace=True)

train = train.merge(y_df[["ID","popularity"]], on="ID", how="left")
train = train.dropna(subset=["popularity"]).reset_index(drop=True)
print(f"   Train: {train.shape} | Test: {test.shape}")

# Align columns
all_feat = [c for c in train.columns if c not in ("ID","popularity")]
for c in set(all_feat) - set(test.columns):
    test[c] = np.nan
for c in set(test.columns) - set(all_feat) - {"ID"}:
    train[c] = np.nan
all_feat = sorted(set(train.columns) - {"ID","popularity"})

# =========================
# 2) FEATURE REDUCTION (redondances)
# =========================
drop_redundant = set()
# MFCCs 6-12
drop_redundant |= {f"mfcc_{i}_{s}" for i in range(6,20) for s in ("mean","std")}
# width/height
drop_redundant |= {"width","height"}
# counts bruts quand log existe
for c in all_feat:
    if re.fullmatch(r".+_count", c) and c.replace("_count","_log_count") in all_feat:
        drop_redundant.add(c)

all_feat = [c for c in all_feat if c not in drop_redundant]
print(f"   Après réduction redondances: {len(all_feat)} features")

# =========================
# 3) BUILD GROUP FEATURE SETS
# =========================
def safe_cols(candidates, df):
    """Garde seulement les colonnes qui existent dans df."""
    return [c for c in candidates if c in df.columns]

onehot_cols = [c for c in all_feat if ONEHOT_COLS_PATTERN.match(c)]
color_xy    = [c for c in all_feat if re.match(r"^[RGBW]\d_[xy]$", c)]

# Groupe 1 : Métadonnées + audio librosa + one-hot + couleurs annexes
GROUP1 = safe_cols(META_COLS + LIBROSA_COLS + onehot_cols + color_xy, train)

# Groupe 2 : Embeddings VGGish + CLIP (après PCA)
GROUP2_VGG  = safe_cols(VGG_COLS,  train)
GROUP2_CLIP = safe_cols(CLIP_COLS, train)

# Groupe 3 : Features vidéo (luminosité + dynamisme + attention)
GROUP3 = safe_cols(VIDEO_COLS, train)

print(f"\n📦 Groupes de features :")
print(f"   Groupe 1 (méta + librosa) : {len(GROUP1)} cols")
print(f"   Groupe 2 VGGish           : {len(GROUP2_VGG)} cols → PCA {PCA_VGG_DIM}")
print(f"   Groupe 2 CLIP             : {len(GROUP2_CLIP)} cols → PCA {PCA_CLIP_DIM}")
print(f"   Groupe 3 (vidéo)          : {len(GROUP3)} cols")

# =========================
# 4) PCA sur embeddings
# =========================
print("\n🔧 PCA embeddings...")
tr_vgg_pca,  te_vgg_pca  = apply_pca(train[GROUP2_VGG].values,  test[GROUP2_VGG].values,  PCA_VGG_DIM,  "vgg")
tr_clip_pca, te_clip_pca = apply_pca(train[GROUP2_CLIP].values, test[GROUP2_CLIP].values, PCA_CLIP_DIM, "clip")

# Matrices finales par groupe
X1_tr = train[GROUP1].fillna(0).reset_index(drop=True)
X1_te = test[GROUP1].fillna(0).reset_index(drop=True)

X2_tr = pd.concat([tr_vgg_pca, tr_clip_pca], axis=1)
X2_te = pd.concat([te_vgg_pca, te_clip_pca], axis=1)

X3_tr = train[GROUP3].fillna(0).reset_index(drop=True)
X3_te = test[GROUP3].fillna(0).reset_index(drop=True)

y = train["popularity"].astype(float).values

# =========================
# 5) LGB PARAMS
# =========================
LGB_PARAMS = {
    "objective":         "regression",
    "metric":            "rmse",
    "learning_rate":     0.05,
    "num_leaves":        31,
    "max_depth":         6,
    "min_child_samples": 30,
    "feature_fraction":  0.5,
    "bagging_fraction":  0.7,
    "bagging_freq":      5,
    "reg_alpha":         0.1,
    "reg_lambda":        1.0,
    "n_jobs":            -1,
    "verbose":           -1,
    "random_state":      SEED,
}

# =========================
# 6) KFOLD STACKING
# =========================
def train_lgb_oof(X_tr, X_te, y, name):
    """Entraîne LGB en KFold, retourne OOF preds + test preds moyennés."""
    kf         = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    oof        = np.zeros(len(X_tr))
    test_preds = np.zeros(len(X_te))
    rmses      = []

    print(f"\n{'─'*50}")
    print(f"  📊 Modèle {name}  |  {X_tr.shape[1]} features")
    print(f"{'─'*50}")

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_tr, y), 1):
        Xf_tr, Xf_val = X_tr.iloc[tr_idx], X_tr.iloc[val_idx]
        yf_tr, yf_val = y[tr_idx], y[val_idx]

        model = lgb.LGBMRegressor(n_estimators=2000, **LGB_PARAMS)
        model.fit(
            Xf_tr, yf_tr,
            eval_set=[(Xf_val, yf_val)],
            callbacks=[
                lgb.early_stopping(100, verbose=False),
                lgb.log_evaluation(period=500),
            ]
        )
        oof[val_idx]    = model.predict(Xf_val)
        test_preds     += model.predict(X_te) / N_SPLITS
        fold_rmse       = np.sqrt(mean_squared_error(yf_val, oof[val_idx]))
        rmses.append(fold_rmse)
        print(f"    Fold {fold} | iter: {model.best_iteration_:4d} | RMSE: {fold_rmse:.4f}")

    oof_rmse = np.sqrt(mean_squared_error(y, oof))
    print(f"  ✅ OOF RMSE {name}: {oof_rmse:.4f}  (mean folds: {np.mean(rmses):.4f})")
    return oof, test_preds, oof_rmse

# Entraînement des 3 modèles de base
oof1, te1, rmse1 = train_lgb_oof(X1_tr, X1_te, y, "LGB-1  méta+librosa")
oof2, te2, rmse2 = train_lgb_oof(X2_tr, X2_te, y, "LGB-2  VGGish+CLIP")
oof3, te3, rmse3 = train_lgb_oof(X3_tr, X3_te, y, "LGB-3  vidéo")

# =========================
# 7) META-MODELE (Ridge)
# =========================
print(f"\n{'='*50}")
print("  🧠 Meta-modèle Ridge")
print(f"{'='*50}")

# Stack OOF pour entraîner le meta-modèle
meta_train = np.column_stack([oof1, oof2, oof3])
meta_test  = np.column_stack([te1,  te2,  te3])

# Cross-val Ridge pour choisir alpha
from sklearn.linear_model import RidgeCV
meta_model = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], cv=5)
meta_model.fit(meta_train, y)

print(f"  Alpha sélectionné : {meta_model.alpha_}")
print(f"  Poids appris      : LGB1={meta_model.coef_[0]:.3f} | LGB2={meta_model.coef_[1]:.3f} | LGB3={meta_model.coef_[2]:.3f}")

final_oof   = meta_model.predict(meta_train)
final_preds = meta_model.predict(meta_test)

final_rmse = np.sqrt(mean_squared_error(y, final_oof))

print(f"\n{'='*50}")
print(f"  OOF RMSE individuel :")
print(f"    LGB-1 méta+librosa : {rmse1:.4f}")
print(f"    LGB-2 VGGish+CLIP  : {rmse2:.4f}")
print(f"    LGB-3 vidéo        : {rmse3:.4f}")
print(f"  ✅ OOF RMSE STACKING : {final_rmse:.4f}")
print(f"{'='*50}")

# =========================
# 8) SUBMISSION
# =========================
submission = pd.DataFrame({
    "ID":         test["ID"].astype(str).values,
    "popularity": final_preds,
})

assert submission["ID"].isna().sum() == 0
assert submission["popularity"].isna().sum() == 0

out_path = ROOT / "submission_stacking.csv"
submission.to_csv(out_path, index=False)

print(f"\n✅ Submission sauvegardée : {out_path}")
print(f"   Shape : {submission.shape}")
print("\nAperçu:")
print(submission.head(10).to_string(index=False))
print("\nStats popularity prédite:")
print(submission["popularity"].describe().round(4))

📂 Chargement...
   Train: (1348, 951) | Test: (338, 950)
   Après réduction redondances: 926 features

📦 Groupes de features :
   Groupe 1 (méta + librosa) : 99 cols
   Groupe 2 VGGish           : 256 cols → PCA 64
   Groupe 2 CLIP             : 512 cols → PCA 64
   Groupe 3 (vidéo)          : 59 cols

🔧 PCA embeddings...
   PCA vgg: 256 → 64 dims | var expliquée: 84.1%
   PCA clip: 512 → 64 dims | var expliquée: 63.0%

──────────────────────────────────────────────────
  📊 Modèle LGB-1  méta+librosa  |  99 features
──────────────────────────────────────────────────
    Fold 1 | iter:   76 | RMSE: 1.2737
    Fold 2 | iter:  210 | RMSE: 1.3828
    Fold 3 | iter:   80 | RMSE: 1.3058
    Fold 4 | iter:  179 | RMSE: 1.2958
    Fold 5 | iter:  150 | RMSE: 1.3572
  ✅ OOF RMSE LGB-1  méta+librosa: 1.3237  (mean folds: 1.3231)

──────────────────────────────────────────────────
  📊 Modèle LGB-2  VGGish+CLIP  |  128 features
──────────────────────────────────────────────────
    Fold 1 | iter: 